In [115]:
import pandas as pd
from generate_xml import load_data, write_xml
import re

In [116]:
data,metadata = load_data('../../zaebuc_written/ZAEBUC-v2.0_release/')
data.head()

/Users/f/Library/CloudStorage/SynologyDrive-ba3sasah/camelLab/zaebuc/corpus_app/src/generate_xml.py:6: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  en = pd.read_csv(f'{datadir}corrected.analyzed_en.tsv',sep='\t',index_col=[0,2,1])


word flag  ... core_pgn pron_pgn
doc_id         Line_Index idx                     ...                  
en-2019-116710 1.0        1    Developments  NaN  ...      NaN      NaN
                          2              in  NaN  ...      NaN      NaN
                          3             the  NaN  ...      NaN      NaN
                          4             UAE  NaN  ...      NaN      NaN
                          5               ,  NaN  ...      NaN      NaN

[5 rows x 13 columns]

In [117]:
import numpy as np
# get arabic mask
isar = data.index.map(lambda x: 'ar' in x[0])

# split available glosses and creaate en gloss_search to ar manual_lemma map
data.loc[isar,'gloss_search'] = data['gloss'].map(lambda x: [x.strip().lower() for x in re.split(r',|;',x)],na_action='ignore')
glossearch = data.reset_index()[['manual_lemma','gloss_search']].explode('gloss_search').replace('',np.nan).drop_duplicates()
gloss_map = glossearch.dropna().groupby('gloss_search')['manual_lemma'].agg(list).to_dict()

# map english lemmas to arabic lemmas when english lemma is in gloss of arabic lemma
data.loc[~isar,'gloss_search'] = data.loc[~isar,'manual_lemma'].map(lambda x: gloss_map.get(x.lower(),[]))


# all glosses include original entry
unique_gloss_lemma = data.copy().apply(lambda x: (tuple(x['gloss_search']),x['manual_lemma']),axis=1,result_type='expand').drop_duplicates(subset=[0,1]).index
data.loc[unique_gloss_lemma].apply(lambda x: x['gloss_search'].append(x['manual_lemma']), axis=1)
print()

In [ ]:
data['gloss_search'][0]

/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_48294/964373190.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  type(data['gloss_search'][0])


list

In [120]:
data.head(2)

word  ...                                  gloss_search
doc_id         Line_Index idx                ...                                              
en-2019-116710 1.0        1    Developments  ...  [تطوير, تنمية, تطور, نمو, نشأة, development]
                          2              in  ...                             [في, منذ, إن, in]

[2 rows x 14 columns]

In [142]:
write_xml(data, metadata, out_path="../data/zaebuc_written.xml", sample=None)
write_xml(data, metadata, out_path="../data/zaebuc_written_sample.xml", sample=100)